# Files in Google Colab: where do my files go?
**MED23CL202 · Python Programming Laboratory** · companion to Experiments 5 and 6 and your mini-project

Your whole mini-project runs in Colab, so you need to know one thing that surprises everyone:

> **Colab runs your code on a temporary Google computer, not on your laptop.** A file you create lives on that computer, and that computer is wiped when your session ends.

In this notebook you will:
1. see where Colab saves your files,
2. write, read and append to a file (modes `"w"`, `"r"`, `"a"`, `"x"`),
3. see which files survive a restart and which disappear,
4. save files in **Google Drive**, so they are still there tomorrow,
5. upload and download files,
6. set up the file handling for **your own mini-project**.

Run each cell with **Shift + Enter**, in order.

## 1. Where am I?
Every Colab notebook starts in a folder called **`/content`** on the temporary computer. A file name with no folder in front of it, like `"patients.txt"`, is saved there.

In [ ]:
import os
print("I am working in:", os.getcwd())
print("Files here:", os.listdir())

👉 Now click the **folder icon** 📁 on the left edge of Colab. That is the **Files panel**: it shows the same `/content` folder. You will see a `sample_data` folder that Colab puts there for you.

## 2. Write mode `"w"`: create a file
`open(name, "w")` creates the file. **If the file already exists, everything in it is erased first.**
`with open(...) as f:` closes the file for you when the block ends, even if your code crashes inside it.

In [ ]:
records = ["Ravi, 98.6", "Meena, 101.2", "Sara, 99.5"]

with open("patients.txt", "w") as f:
    for r in records:
        f.write(r + "\n")          # \n ends the line; write() does not add it for you

print("Written. Look for patients.txt in the Files panel (click the refresh icon if needed).")

## 3. Read mode `"r"`: read it back
Three common ways to read:

In [ ]:
with open("patients.txt", "r") as f:
    whole = f.read()               # the whole file as ONE string
print(repr(whole))

with open("patients.txt", "r") as f:
    lines = f.readlines()          # a LIST of lines, each still ending in \n
print(lines)

with open("patients.txt", "r") as f:
    for line in f:                 # one line at a time (best for big files)
        name, temp = line.strip().split(", ")
        print(name, "->", float(temp))

## 4. Append mode `"a"`: add without erasing
`"a"` adds to the **end** of the file and keeps what is already there.

**Try it:** run the next cell **three times**, then run the cell after it.

In [ ]:
with open("patients.txt", "a") as f:
    f.write("John, 97.9\n")
print("One line added.")

In [ ]:
with open("patients.txt", "r") as f:
    print(f.read())

Now compare with `"w"`: run this cell, then read the file again (run the cell above).

In [ ]:
with open("patients.txt", "w") as f:
    f.write("Only this line is left!\n")
print("Overwritten with 'w'.")

| Mode | If the file does not exist | If the file exists | Use it to |
|---|---|---|---|
| `"r"` | **FileNotFoundError** | reads it | read saved data |
| `"w"` | creates it | **erases it**, then writes | start a fresh file |
| `"a"` | creates it | adds to the end | keep adding records |
| `"x"` | creates it | **FileExistsError** | create a file only once, never overwrite |

The `"x"` mode in action:

In [ ]:
try:
    with open("settings.txt", "x") as f:
        f.write("created once\n")
    print("settings.txt created.")
except FileExistsError:
    print("settings.txt already exists, so 'x' refused to overwrite it.")

Run that cell twice: the first time it creates the file, the second time it refuses.

And reading a file that is not there:

In [ ]:
try:
    with open("no_such_file.txt", "r") as f:
        print(f.read())
except FileNotFoundError:
    print("FileNotFoundError: check the name, the folder, and whether the file was lost (section 6).")

## 5. Any kind of file works
Your project can use **any** file you like: a plain text file, a CSV, a JSON file. Here are two common ones.

**CSV** (a table that Excel and pandas can open):

In [ ]:
import csv

rows = [["name", "temp"], ["Ravi", 98.6], ["Meena", 101.2]]
with open("vitals.csv", "w", newline="") as f:     # newline="" stops blank lines appearing between rows
    csv.writer(f).writerows(rows)

with open("vitals.csv", "a", newline="") as f:     # append one more row
    csv.writer(f).writerow(["Sara", 99.5])

with open("vitals.csv", "r") as f:
    reader = csv.reader(f)
    header = next(reader)                          # skip the header row
    for name, temp in reader:
        print(name, float(temp))

**JSON** (handy for saving a whole dictionary or list in one go):

In [ ]:
import json

stock = {"Paracetamol": 120, "Amoxicillin": 15}
with open("stock.json", "w") as f:
    json.dump(stock, f)

with open("stock.json", "r") as f:
    loaded = json.load(f)
print(loaded, type(loaded))

## 6. Temporary files: what survives and what is lost
Everything so far is in `/content`, on the **temporary** computer.

| What you do | Your files in `/content` |
|---|---|
| Re-run cells | stay ✅ |
| **Runtime → Restart session** | stay ✅ (your *variables* are cleared, the files are not) |
| **Runtime → Disconnect and delete runtime** | **lost** ❌ |
| Close the tab and come back later, or leave Colab idle for a long time | **lost** ❌ (Colab disconnects idle sessions and also has a maximum session length; the exact times are set by Google and can change) |

**Try it now:**
1. Run the cell below and note the time it prints.
2. Choose **Runtime → Restart session**, then run the cell after it. The file is still there.
3. Choose **Runtime → Disconnect and delete runtime**, reconnect, then run the second cell again. The file is gone.

In [ ]:
from datetime import datetime
with open("session_test.txt", "w") as f:
    f.write("written at " + datetime.now().strftime("%H:%M:%S"))
print("session_test.txt written at", datetime.now().strftime("%H:%M:%S"))

In [ ]:
import os
if os.path.exists("session_test.txt"):
    with open("session_test.txt") as f:
        print("Still here:", f.read())
else:
    print("session_test.txt is gone: the temporary computer was wiped.")

**So when is a temporary file the right choice?** For things you only need during this session: a test file, a scratch copy, a chart image you will download right away. It is fast and needs no permission.

**When is it the wrong choice?** For your project's records. If you save patient readings in `/content` today, they will be gone tomorrow.

## 7. Google Drive: files that survive across sessions
Connect ("mount") your own Google Drive. Colab will ask you to allow access the first time; choose your college Google account and allow it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Your Drive now appears inside Colab at **`/content/drive/MyDrive`**. You can see it in the Files panel too.

Make a folder for your project and save a file there:

In [ ]:
import os

DATA_DIR = "/content/drive/MyDrive/MiniProject"
os.makedirs(DATA_DIR, exist_ok=True)               # creates the folder; no error if it already exists

drive_file = os.path.join(DATA_DIR, "visits.txt")
with open(drive_file, "a") as f:                   # "a": every run adds one line
    f.write("visit recorded\n")

with open(drive_file) as f:
    print(f.read())
print("Saved in your Google Drive:", drive_file)

**Try it:**
1. Open [drive.google.com](https://drive.google.com) in a new tab: you will find a **MiniProject** folder with `visits.txt` inside.
2. Choose **Runtime → Disconnect and delete runtime**, reconnect, run the `drive.mount` cell again, then run the cell above. The old lines are **still there**, and one more is added.

Two things to remember:
- **Mount Drive again in every new session.** After a disconnect, `/content/drive` is empty until you run `drive.mount` again.
- **Changes reach Drive within a short while.** If you are about to close the notebook right after saving, first run `drive.flush_and_unmount()` so every change is written to Drive. (After that, mount again before using Drive files.)

## 8. Uploading and downloading
**Upload** a file from your laptop (for example a CSV you were given). It goes into `/content`, so it is **temporary**; copy it to Drive if you need it again.

In [ ]:
from google.colab import files
import shutil, os

uploaded = files.upload()                          # opens a "Choose files" button
project_dir = "/content/drive/MyDrive/MiniProject"
for name in uploaded:
    print("Uploaded", name, "to /content (temporary)")
    if os.path.isdir(project_dir):                 # Drive is mounted (section 7)
        shutil.copy(name, os.path.join(project_dir, name))
        print("Copied to Drive as well:", os.path.join(project_dir, name))
    else:
        print("Drive is not mounted, so there is no permanent copy. Run the drive.mount cell first.")

**Download** a file to your laptop:

In [ ]:
from google.colab import files

with open("my_readings.txt", "w") as f:            # make a small file to download
    f.write("Ravi, 98.6\nMeena, 101.2\n")
files.download("my_readings.txt")                  # your browser saves it

You can also right-click any file in the Files panel and choose **Download**, or drag a file from your laptop into the panel to upload it.

## 9. Common problems
| What you see | Why | Fix |
|---|---|---|
| `FileNotFoundError` for a file you made yesterday | it was in `/content` and the runtime was wiped | save project files in Drive (section 7) |
| `FileNotFoundError` for `/content/drive/...` | Drive is not mounted in this session | run `drive.mount("/content/drive")` first |
| Your old records disappeared | you opened the file with `"w"` instead of `"a"` | use `"a"` to add; use `"w"` only to start fresh |
| All the text is on one line | `write()` does not add a new line | add `"\n"` at the end of each line |
| Blank lines between CSV rows | missing `newline=""` | `open(name, "w", newline="")` |
| The new file is not in the Files panel | the panel does not refresh by itself | click the refresh icon in the Files panel |

**Paths are just folders and names.** `"vitals.csv"` means `/content/vitals.csv` (temporary). `"/content/drive/MyDrive/MiniProject/vitals.csv"` means a file in your Drive (permanent).

## 10. Set up your mini-project
Copy these two cells to the **top** of your mini-project notebook and change the file name to suit your project.
- Name your notebook with your register number, e.g. **`E0125001_miniproject.ipynb`** (click the name at the top left). Keep that title bar visible in your screenshots for the report.
- Your data file can be **any** kind (text, CSV, JSON) and you can use **any** mode that fits your program.

In [ ]:
# ---- Cell 1: connect Google Drive (run this first in every session) ----
from google.colab import drive
drive.mount("/content/drive")

import os
DATA_DIR = "/content/drive/MyDrive/MiniProject"     # your project folder in Drive
os.makedirs(DATA_DIR, exist_ok=True)
DATA_FILE = os.path.join(DATA_DIR, "vitals.txt")    # change the name to suit your project
print("Project data file:", DATA_FILE)

In [ ]:
# ---- Cell 2: an example of loading and saving records (adapt it to your project) ----
def load_records():
    """Read all saved records; an empty list if nothing has been saved yet."""
    if not os.path.exists(DATA_FILE):
        return []
    with open(DATA_FILE, "r") as f:
        return [line.strip() for line in f if line.strip()]

def save_record(text):
    """Add one record to the end of the file."""
    with open(DATA_FILE, "a") as f:
        f.write(text + "\n")

save_record("Ravi, 98.6")
print("Records so far:", load_records())

**For your report:** take a screenshot of your program saving something to its file, and of it reading the file back if your project does that. You can also open the file from the Files panel (or from Drive) and screenshot its contents.

### Check yourself
1. You saved `log.txt` with no folder in front of the name. Where is it, and will it be there tomorrow?
2. Which mode would you use to add today's readings without losing yesterday's?
3. You disconnected and reconnected. Your Drive file opens with `FileNotFoundError`. What did you forget?
4. What is the difference between **Restart session** and **Disconnect and delete runtime** for your files?